Primeira prototipagem de full-load ao GCS

In [6]:
#CELULA 1: Imports, Variaveis de Ambiente, GCP Client e Configuracoes Globais

import os
import io
import re
import zipfile
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client do Google Cloud Storage
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Gerais do Pipeline e Buckets
ANO_ALVO = "2026"
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"
BUCKET_SILVER = "cvm-lakehouse-silver-dev"
MES_ATUAL = datetime.now().month

# 4. Configuração de Exibição do Pandas
pd.set_option("display.max_columns", None)

print(f"✔ Ambiente GCP autenticado com sucesso!")
print(f"✔ Target Bronze Bucket: gs://{BUCKET_BRONZE}")
print(f"✔ Target Silver Bucket: gs://{BUCKET_SILVER}")
print(f"✔ Ano de Referência: {ANO_ALVO}")

✔ Ambiente GCP autenticado com sucesso!
✔ Target Bronze Bucket: gs://cvm-landing-zone-bronze-dev
✔ Target Silver Bucket: gs://cvm-lakehouse-silver-dev
✔ Ano de Referência: 2026


In [ ]:
#CELULA 2: Client GCP e Auxiliar de Streaming In-Memory

client = storage.Client.from_service_account_json(KEY_PATH)
bucket = client.bucket(BUCKET_BRONZE)

def upload_in_memory(url: str, gcs_path: str, content_type: str) -> bool:
    """
    Realiza o download streaming via requests e envia diretamente para o GCS em memória.
    Retorna True se o upload foi bem-sucedido.
    """
    try:
        response = requests.get(url, timeout=90)
        if response.status_code == 200:
            blob = bucket.blob(gcs_path)
            blob.upload_from_string(response.content, content_type=content_type)
            print(f" Success: gs://{BUCKET_BRONZE}/{gcs_path}")
            return True
        elif response.status_code == 404:
            print(f" Arquivo não encontrado na origem (HTTP 404): {url}")
            return False
        else:
            print(f" Erro HTTP {response.status_code} para {url}")
            return False
    except Exception as e:
        print(f" Exceção ao processar {url}: {str(e)}")
        return False

In [ ]:
#CELULA 3: Engine do Check Reverso (Tabela Calendario vs. Landing Zone)

def gerar_tabela_calendario_dias_uteis(ano: str) -> list:
    """Gera a lista de datas em dias úteis (segunda a sexta) do início do ano até ontem no formato YYYYMMDD."""
    hoje = datetime.now().date()
    ontem = hoje - timedelta(days=1)
    dias_uteis = pd.bdate_range(start=f"{ano}-01-01", end=ontem)
    return dias_uteis.strftime("%Y%m%d").tolist()

def obter_datas_faltantes_gcs(prefixo_gcs: str, sufixo_extensao: str = ".csv", ano_alvo: str = "2026") -> list:
    """
    Compara os arquivos existentes no GCS com a Tabela Calendário de Dias Úteis 
    e retorna a lista exata de datas (YYYYMMDD) pendentes de ingestão.
    """
    datas_esperadas = set(gerar_tabela_calendario_dias_uteis(ano_alvo))
    blobs = client.list_blobs(BUCKET_BRONZE, prefix=prefixo_gcs)
    
    datas_existentes = set()
    padrao_data = re.compile(r"(\d{8})")
    
    for blob in blobs:
        if blob.name.endswith(sufixo_extensao):
            match = padrao_data.search(blob.name)
            if match:
                datas_existentes.add(match.group(1))
                
    datas_faltantes = sorted(list(datas_esperadas - datas_existentes))
    
    print(f"\n--- DIAGNÓSTICO DE INGESTÃO [{prefixo_gcs}] ---")
    print(f"✔ Dias úteis esperados em {ano_alvo}: {len(datas_esperadas)}")
    print(f"✔ Arquivos na Landing Zone (GCS): {len(datas_existentes)}")
    print(f"⚠ Datas pendentes para download: {len(datas_faltantes)}")
    
    return datas_faltantes

In [ ]:
#CELULA 4: Ingestao Incremental - Cadastro de Fundos (CAD_FI)

print("\n=== INGESTÃO INCREMENTAL: CADASTRO DE FUNDOS (CAD_FI) ===")

datas_cad_pendentes = obter_datas_faltantes_gcs(
    prefixo_gcs=f"cvm/cad_fi/ano={ANO_ALVO}/", 
    sufixo_extensao=".csv",
    ano_alvo=ANO_ALVO
)

if not datas_cad_pendentes:
    print("✨ [CAD_FI] Landing Zone 100% atualizada para todos os dias úteis!")
else:
    print(f"🚀 Baixando foto cadastral para {len(datas_cad_pendentes)} data(s) pendente(s)...")
    url_cad = "http://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
    
    for data_str in datas_cad_pendentes:
        mes_str = data_str[4:6]
        path_cad = f"cvm/cad_fi/ano={ANO_ALVO}/mes={mes_str}/cad_fi_{data_str}.csv"
        upload_in_memory(url_cad, path_cad, "text/csv")

In [ ]:
#CELULA 5: Ingestao Mensal - Informe Diario e CDA_FI (CVM)

print(f"\n=== INGESTÃO BRONZE: INFORME DIÁRIO E CARTEIRA (CDA_FI) ===")

for mes in range(1, MES_ATUAL + 1):
    mes_str = f"{mes:02d}"
    ano_mes = f"{ANO_ALVO}{mes_str}"
    
    # 1. Informe Diário
    url_inf = f"http://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ano_mes}.zip"
    path_inf = f"cvm/inf_diario_fi/ano={ANO_ALVO}/mes={mes_str}/inf_diario_fi_{ano_mes}.zip"
    upload_in_memory(url_inf, path_inf, "application/zip")
    
    # 2. Carteira Mensal (CDA_FI)
    url_cda = f"http://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ano_mes}.zip"
    path_cda = f"cvm/cda_fi/ano={ANO_ALVO}/mes={mes_str}/cda_fi_{ano_mes}.zip"
    upload_in_memory(url_cda, path_cda, "application/zip")

In [ ]:
#CELULA 6: Ingestao do Benchmark (API BCB - SGS Serie 12 / CDI)

def ingest_bcb_cdi_bronze():
    print(f"\n=== INGESTÃO BRONZE: BCB CDI (SGS 12) ===")
    data_inicial = f"01/01/{ANO_ALVO}"
    data_final = datetime.now().strftime("%d/%m/%Y")
    
    url_bcb = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"
    
    response = requests.get(url_bcb, timeout=30)
    if response.status_code == 200:
        gcs_path = f"bcb/cdi/ano={ANO_ALVO}/cdi_diario_{ANO_ALVO}.json"
        blob = bucket.blob(gcs_path)
        blob.upload_from_string(response.content, content_type="application/json")
        print(f" Upload realizado com sucesso: gs://{BUCKET_BRONZE}/{gcs_path}")
    else:
        print(f" Erro ao acessar API do BCB. Status Code: {response.status_code}")

ingest_bcb_cdi_bronze()

In [ ]:
#CELULA 7: Prototipagem Camada Silver - Aplicacao dos 5 Filtros de Credito Privado (CDA_FI)

print(f"\n=== PROTOTIPAGEM SILVER: FILTROS DE CRÉDITO PRIVADO ===")

# Teste com amostragem de um mês para validação dos filtros
ano_mes_teste = "202401"
url_cda_teste = f"http://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ano_mes_teste}.zip"
res_cda = requests.get(url_cda_teste, timeout=60)

if res_cda.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(res_cda.content)) as z:
        # 1. PL dos Fundos
        df_pl = pd.read_csv(z.open(f'cda_fi_PL_{ano_mes_teste}.csv'), sep=";", encoding="ISO-8859-1")
        df_pl = df_pl[['CNPJ_FUNDO_CLASSE', 'DENOM_SOCIAL', 'VL_PATRIM_LIQ']].drop_duplicates()
        df_pl = df_pl[df_pl['VL_PATRIM_LIQ'] > 0]

        # 2. BLC_2: FIDC
        df_blc2 = pd.read_csv(z.open(f'cda_fi_BLC_2_{ano_mes_teste}.csv'), sep=";", encoding="ISO-8859-1")
        fidc_pos = df_blc2[df_blc2['TP_ATIVO'].str.contains('FIDC', na=False, case=False)]
        fidc_sum = fidc_pos.groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index().rename(columns={'VL_MERC_POS_FINAL': 'VL_FIDC'})

        # 3. BLC_4: Ações
        df_blc4 = pd.read_csv(z.open(f'cda_fi_BLC_4_{ano_mes_teste}.csv'), sep=";", encoding="ISO-8859-1")
        acoes_pos = df_blc4[df_blc4['TP_ATIVO'].str.contains('AÇÕES|ACOES|UNIT', na=False, case=False)]
        acoes_sum = acoes_pos.groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index().rename(columns={'VL_MERC_POS_FINAL': 'VL_ACOES'})

        # 4. BLC_5 + BLC_6: Crédito Privado
        df_blc5 = pd.read_csv(z.open(f'cda_fi_BLC_5_{ano_mes_teste}.csv'), sep=";", encoding="ISO-8859-1")
        df_blc6 = pd.read_csv(z.open(f'cda_fi_BLC_6_{ano_mes_teste}.csv'), sep=";", encoding="ISO-8859-1")
        credito_sum = pd.concat([df_blc5, df_blc6]).groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index().rename(columns={'VL_MERC_POS_FINAL': 'VL_CREDITO'})

    # Consolidando matriz
    df_mestre = df_pl.merge(fidc_sum, on='CNPJ_FUNDO_CLASSE', how='left') \
                     .merge(acoes_sum, on='CNPJ_FUNDO_CLASSE', how='left') \
                     .merge(credito_sum, on='CNPJ_FUNDO_CLASSE', how='left').fillna(0)

    # Métricas % do PL
    df_mestre['PCT_CREDITO'] = df_mestre['VL_CREDITO'] / df_mestre['VL_PATRIM_LIQ']
    df_mestre['PCT_FIDC'] = df_mestre['VL_FIDC'] / df_mestre['VL_PATRIM_LIQ']
    df_mestre['PCT_ACOES'] = df_mestre['VL_ACOES'] / df_mestre['VL_PATRIM_LIQ']

    # Regras de Elegibilidade (5 Filtros)
    cond_credito = df_mestre['PCT_CREDITO'] >= 0.10
    cond_nome = ~df_mestre['DENOM_SOCIAL'].str.contains('INVESTIMENTO NO EXTERIOR| IE | EXTERIOR', case=False, regex=True, na=False)
    cond_fidc = df_mestre['PCT_FIDC'] <= 0.30
    cond_acoes = df_mestre['PCT_ACOES'] <= 0.03

    df_fundos_credito_elegiveis = df_mestre[cond_credito & cond_nome & cond_fidc & cond_acoes]

    print(f"✔ Fundos analisados: {len(df_mestre)}")
    print(f"✔ Fundos de Crédito Privado Elegíveis: {len(df_fundos_credito_elegiveis)}")

In [ ]:
#CELULA 8: Visualizaçao dos Fundos Elegiveis Filtrados

df_fundos_credito_elegiveis[['CNPJ_FUNDO_CLASSE', 'DENOM_SOCIAL', 'PCT_CREDITO', 'PCT_FIDC', 'PCT_ACOES']].head(10)